In [0]:
%run ../configs/log_configs

In [ ]:
# Databricks notebook source
# ============================================
# BIB Silver Layer Formation Notebook
# Creates dimensions + facts from bronze
# ============================================


from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from delta.tables import DeltaTable
from datetime import datetime

# --------------------------------------------
# Widgets
# --------------------------------------------
dbutils.widgets.text("environment", "dev")
dbutils.widgets.text("layer", "silver")
dbutils.widgets.text("notebook_name", "02_silver_layer_modeling")
dbutils.widgets.text("pipeline_name", "BIB_Data_Processing")

environment   = dbutils.widgets.get("environment")
layer         = dbutils.widgets.get("layer")
notebook_name = dbutils.widgets.get("notebook_name")
pipeline_name = dbutils.widgets.get("pipeline_name")

# --------------------------------------------
# Standard helpers
# --------------------------------------------
def bronze_table_name(source_name: str) -> str:
    return f"bib_catalog.bronze.{source_name}"

def silver_table_name(entity_name: str) -> str:
    return f"bib_catalog.silver.{entity_name}"

def table_exists(table_name: str) -> bool:
    return spark.catalog.tableExists(table_name)

def present_cols(df, cols):
    return [c for c in cols if c in df.columns]

def rename_if_present(df, rename_map):
    for old_name, new_name in rename_map.items():
        if old_name in df.columns and old_name != new_name:
            df = df.withColumnRenamed(old_name, new_name)
    return df

def trim_string_cols(df):
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(field.name, F.trim(F.col(field.name)))
            df = df.withColumn(
                field.name,
                F.when(F.col(field.name) == "", None).otherwise(F.col(field.name))
            )
    return df

def safe_cast(df, col_name, target_type):
    target_type = target_type.lower()
    if col_name not in df.columns:
        return df

    if target_type in ["double", "integer", "long", "date", "timestamp", "boolean"]:
        return df.withColumn(col_name, F.expr(f"try_cast(`{col_name}` as {target_type})"))
    return df.withColumn(col_name, F.col(col_name).cast(target_type))

def cast_columns(df, cast_map):
    for col_name, col_type in cast_map.items():
        df = safe_cast(df, col_name, col_type)
    return df

def add_audit_cols(df, source_name):
    return (
        df
        .withColumn("silver_updated_at", F.current_timestamp())
        .withColumn("silver_source", F.lit(source_name))
        .withColumn("silver_environment", F.lit(environment))
    )

def select_clean_transform(df, cfg):
    """
    1) rename columns
    2) keep only required columns
    3) trim strings
    4) cast types
    5) drop duplicates if configured
    """
    df = rename_if_present(df, cfg.get("rename_map", {}))
    keep_cols = present_cols(df, cfg["keep_cols"])
    df = df.select(*keep_cols)
    df = trim_string_cols(df)
    df = cast_columns(df, cfg.get("cast_map", {}))

    if cfg.get("primary_key"):
        df = df.dropDuplicates([cfg["primary_key"]])

    return df

def with_date_key(df, ts_col, date_key_col="date_key"):
    """
    Creates an integer date key in yyyyMMdd format.
    """
    if ts_col in df.columns:
        return df.withColumn(date_key_col, F.date_format(F.to_date(F.col(ts_col)), "yyyyMMdd").cast("int"))
    return df.withColumn(date_key_col, F.lit(None).cast("int"))

# --------------------------------------------
# Delta writers
# --------------------------------------------
def write_overwrite(df, target_table, adls_path=None, partition_col=None):
    writer = (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )
    if partition_col and partition_col in df.columns:
        writer = writer.partitionBy(partition_col)
    writer.saveAsTable(target_table)

    if adls_path:
        adls_writer = (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
        )
        if partition_col and partition_col in df.columns:
            adls_writer = adls_writer.partitionBy(partition_col)
        adls_writer.save(adls_path)

def merge_upsert(df_new, target_table, primary_key, adls_path=None, partition_col=None):
    if not table_exists(target_table):
        write_overwrite(df_new, target_table, adls_path, partition_col)
        return

    # Schema drift check — if column sets differ (e.g. SCD columns left behind,
    # or new columns added to source), fall back to a full overwrite so the merge
    # UPDATE clause never references a column that doesn't exist in the target.
    target_cols  = set(spark.table(target_table).columns)
    source_cols  = set(df_new.columns)
    if target_cols != source_cols:
        added    = source_cols - target_cols
        removed  = target_cols - source_cols
        print(f"  Schema drift detected on {target_table}")
        if added:
            print(f"    New columns in source   : {sorted(added)}")
        if removed:
            print(f"    Stale columns in target : {sorted(removed)}")
        print(f"  Falling back to full overwrite to realign schema.")
        write_overwrite(df_new, target_table, adls_path, partition_col)
        return

    delta_table = DeltaTable.forName(spark, target_table)

    audit_cols = {
        primary_key,
        "silver_updated_at",
        "silver_source",
        "silver_environment"
    }
    update_cols = {
        c: f"source.{c}"
        for c in df_new.columns
        if c not in audit_cols
    }

    (
        delta_table.alias("target")
        .merge(
            df_new.alias("source"),
            f"target.{primary_key} = source.{primary_key}"
        )
        .whenMatchedUpdate(set=update_cols)
        .whenNotMatchedInsertAll()
        .execute()
    )

    if adls_path:
        (
            DeltaTable.forName(spark, target_table)
            .toDF()
            .write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .save(adls_path)
        )

# --------------------------------------------
# Silver table configs
# --------------------------------------------
TABLES = {
    # --------------------------
    # Dimensions
    # --------------------------

    # =====================================================
    # DIM CUSTOMER
    # =====================================================
    "dim_customer": {

        "source": "customers",
        "kind": "dim",
        "primary_key": "customer_id",

        "keep_cols": [
            "customer_id",
            "full_name",
            "email",
            "phone",
            "date_of_birth",
            "gender",
            "city",
            "state",
            "pin_code",
            "occupation",
            "income_band",
            "segment",
            "kyc_status",
            "branch_id",
            "manager_id",
            "onboarded_date"
        ],

        "rename_map": {
            "segment": "customer_segment"
        },

        "cast_map": {
            "date_of_birth": "date",
            "onboarded_date": "date"
        }
    },

    # =====================================================
    # DIM ACCOUNT
    # =====================================================
    "dim_account": {

        "source": "accounts",
        "kind": "dim",
        "primary_key": "account_id",

        "keep_cols": [
            "account_id",
            "customer_id",
            "account_type",
            "product_id",
            "balance",
            "status",
            "opened_date",
            "closed_date"
        ],

        "rename_map": {
            "status": "account_status",
            "opened_date": "open_date",
            "closed_date": "close_date"
        },

        "cast_map": {
            "balance": "double",
            "open_date": "date",
            "close_date": "date"
        }
    },

    # =====================================================
    # DIM PRODUCT
    # =====================================================
    "dim_product": {

        "source": "products",
        "kind": "dim",
        "primary_key": "product_id",

        "keep_cols": [
            "product_id",
            "product_name",
            "category",
            "sub_category",
            "description",
            "min_income_required",
            "eligibility_criteria",
            "is_active"
        ],

        "rename_map": {
            "category": "product_category",
            "sub_category": "product_sub_category",
            "is_active": "active_flag"
        },

        "cast_map": {
            "min_income_required": "double"
        }
    },

    # =====================================================
    # DIM BRANCH
    # =====================================================
    "dim_branch": {

        "source": "branches",
        "kind": "dim",
        "primary_key": "branch_id",

        "keep_cols": [
            "branch_id",
            "branch_name",
            "branch_code",
            "city",
            "state",
            "zone",
            "region",
            "branch_type"
        ],

        "rename_map": {},

        "cast_map": {}
    },

    # =====================================================
    # DIM MANAGER
    # =====================================================
    "dim_manager": {

        "source": "bank_managers",
        "kind": "dim",
        "primary_key": "manager_id",

        "keep_cols": [
            "manager_id",
            "full_name",
            "email",
            "role",
            "branch_id",
            "zone",
            "region",
            "joining_date",
            "is_active"
        ],

        "rename_map": {
            "full_name": "manager_name",
            "role": "designation",
            "joining_date": "hire_date",
            "is_active": "manager_status"
        },

        "cast_map": {
            "hire_date": "date"
        }
    },

    # =====================================================
    # FACT TRANSACTIONS
    # =====================================================
    "fact_transactions": {

        "source": "transactions",
        "kind": "fact",
        "primary_key": "txn_id",

        "keep_cols": [
            "txn_id",
            "customer_id",
            "account_id",
            "amount",
            "txn_type",
            "channel",
            "merchant_category",
            "txn_datetime",
            "status"
        ],

        "rename_map": {
            "channel": "txn_channel",
            "status": "txn_status"
        },

        "cast_map": {
            "txn_datetime": "timestamp",
            "amount": "double"
        }
    },

    # =====================================================
    # FACT DIGITAL ACTIVITY
    # =====================================================
    "fact_digital_activity": {

        "source": "digital_activity",
        "kind": "fact",
        "primary_key": "activity_id",

        "keep_cols": [
            "activity_id",
            "customer_id",
            "event_datetime",
            "event_type",
            "channel",
            "page_or_feature",
            "session_id",
            "device_type"
        ],

        "rename_map": {
            "page_or_feature": "page_name"
        },

        "cast_map": {
            "event_datetime": "timestamp"
        }
    },

    # =====================================================
    # FACT CAMPAIGN INTERACTIONS
    # =====================================================
    "fact_campaign_interactions": {

        "source": "campaign_interactions",
        "kind": "fact",
        "primary_key": "interaction_id",

        "keep_cols": [
            "interaction_id",
            "campaign_id",
            "campaign_name",
            "product_id",
            "customer_id",
            "action_type",
            "interaction_datetime",
            "converted"
        ],

        "rename_map": {},

        "cast_map": {
            "interaction_datetime": "timestamp",
            "converted": "integer"
        }
    },

    # =====================================================
    # FACT PRODUCT HOLDINGS
    # =====================================================
    "fact_product_holdings_snapshot": {

        "source": "product_holdings",
        "kind": "fact",
        "primary_key": "holding_id",

        "keep_cols": [
            "holding_id",
            "customer_id",
            "product_id",
            "status",
            "start_date",
            "end_date",
            "value_amount"
        ],

        "rename_map": {
            "status": "holding_status"
        },

        "cast_map": {
            "start_date": "date",
            "end_date": "date",
            "value_amount": "double"
        }
    }
}

# --------------------------------------------
# Main processing logic
# --------------------------------------------
def process_table(entity_name, cfg):
    source_name  = cfg["source"]
    bronze_table  = bronze_table_name(source_name)
    silver_table  = silver_table_name(entity_name)
    adls_path     = f"abfss://{environment}@{storage_account}.dfs.core.windows.net/silver/{entity_name}/"

    print(f"\n{'='*80}")
    print(f"Processing {source_name} -> {silver_table}")
    print(f"{'='*80}")

    # Read Bronze
    df = spark.read.table(bronze_table)

    # Remove Bronze audit columns
    audit_cols = [
        "bib_ingested_at",
        "bib_source",
        "bib_environment",
        "bib_is_deleted"
    ]

    df = df.drop(*[c for c in audit_cols if c in df.columns])

    # Transform
    df = select_clean_transform(df, cfg)

    # Add derived date_key for facts where relevant
    if entity_name == "fact_transactions" and "txn_datetime" in df.columns:
        df = with_date_key(df, "txn_datetime")
        df = df.withColumn("txn_date", F.to_date("txn_datetime"))

    elif entity_name == "fact_digital_activity" and "event_datetime" in df.columns:
        df = with_date_key(df, "event_datetime")
        df = df.withColumn("event_date", F.to_date("event_datetime"))

    elif entity_name == "fact_campaign_interactions" and "interaction_datetime" in df.columns:
        df = with_date_key(df, "interaction_datetime")
        df = df.withColumn("interaction_date", F.to_date("interaction_datetime"))

    elif entity_name == "fact_product_holdings_snapshot" and "start_date" in df.columns:
        df = with_date_key(df, "start_date")

    # Add audit cols
    df = add_audit_cols(df, source_name)

    # Write to silver — all tables use upsert (merge by primary key)
    merge_upsert(
        df_new=df,
        target_table=silver_table,
        primary_key=cfg["primary_key"],
        adls_path=adls_path
    )

    print(f"Completed: {entity_name}")

# --------------------------------------------
# Run all silver entities
# --------------------------------------------
for entity_name, cfg in TABLES.items():
    process_table(entity_name, cfg)

print("\nAll silver tables processed successfully.")

In [0]:
# # ==========================================
# # OPTIMIZE + ZORDER Helper
# # ==========================================

# def optimize_silver_table(table_name, zorder_cols=None):
#     """
#     Optimizes Delta table.
#     Optionally applies ZORDER.
#     """

#     try:

#         if zorder_cols:

#             if isinstance(zorder_cols, list):
#                 cols = ", ".join(zorder_cols)
#             else:
#                 cols = zorder_cols

#             print(f"OPTIMIZE {table_name} ZORDER BY ({cols})")

#             spark.sql(f"""
#                 OPTIMIZE {table_name}
#                 ZORDER BY ({cols})
#             """)

#         else:

#             print(f"OPTIMIZE {table_name}")

#             spark.sql(f"""
#                 OPTIMIZE {table_name}
#             """)

#         print(f"Completed : {table_name}\n")

#     except Exception as e:

#         print(f"Failed : {table_name}")
#         print(str(e))

In [ ]:
# tables = spark.sql(
#     "SHOW TABLES IN bib_catalog.silver"
# ).collect()

# for row in tables:

#     table = f"bib_catalog.silver.{row.tableName}"

#     print(f"Optimizing {table}")

#     spark.sql(f"""
#         OPTIMIZE {table}
    """)

In [0]:
# %sql
# SELECT COUNT(*), TO_DATE(bib_ingested_at) as ingested_date
# FROM bib_catalog.bronze.transactions
# GROUP BY TO_DATE(bib_ingested_at)
# ORDER BY ingested_date DESC

In [0]:
%sql
CREATE OR REPLACE FUNCTION
bib_catalog.silver.mask_phone(phone STRING)
RETURNS STRING
RETURN
    CASE
        WHEN phone IS NULL THEN NULL
        ELSE concat('XXXXXX', right(phone, 4))
    END;

ALTER TABLE bib_catalog.silver.customers
ALTER COLUMN phone
SET MASK bib_catalog.silver.mask_phone;

In [0]:
%sql
CREATE OR REPLACE FUNCTION
bib_catalog.silver.mask_email(email STRING)
RETURNS STRING
RETURN
    CASE
        WHEN email IS NULL THEN NULL
        ELSE concat(left(email, 2), '****@masked.com')
    END;

ALTER TABLE bib_catalog.silver.customers
ALTER COLUMN email
SET MASK bib_catalog.silver.mask_email;

In [0]:
%sql
CREATE OR REPLACE FUNCTION
bib_catalog.silver.mask_pin_code(pin_code STRING)
RETURNS STRING
RETURN
    CASE
        WHEN pin_code IS NULL THEN NULL
        ELSE concat(left(pin_code, 2), 'XXXX')
    END;

ALTER TABLE bib_catalog.silver.customers
ALTER COLUMN pin_code
SET MASK bib_catalog.silver.mask_pin_code;

In [0]:
# %sql
# ALTER TABLE bib_catalog.silver.customers
# ALTER COLUMN phone
# DROP MASK;

In [0]:
# %sql
# ALTER TABLE bib_catalog.silver.customers
# ALTER COLUMN email
# DROP MASK;

In [0]:
# %sql
# ALTER TABLE bib_catalog.silver.customers
# ALTER COLUMN pin_code
# DROP MASK;